> # Instructor / solution version
> Complete reference implementation. Accuracy and latency remain empirical.

# Efficient Edge AI - MNIST hands-on with TensorFlow/Keras

**Goal:** compare a dense Multi-Layer Perceptron (MLP) with a small CNN on **MNIST** handwritten digits. Both models classify grayscale $28\times28$ images into the digits 0-9.

You will implement and train both models, then compare training/validation curves, test accuracy, parameter count and memory, MACs per image, and batch-1 inference latency.

## Architectures

**Dense MLP:** `28×28×1 -> Flatten -> Dense(128) -> Dense(64) -> Dense(10)`

**Small CNN:** `28×28×1 -> Conv3×3(8) -> MaxPool2 -> Conv3×3(16) -> MaxPool2 -> Flatten -> Dense(32) -> Dense(10)`

Use ReLU after hidden Conv/Dense layers. The last layer outputs **logits**, not probabilities. The intentionally small architectures make the comparison suitable for edge AI.

Suggested time: **45-60 min**.

## 0. Esecuzione in Google Colab

[Apri Google Colab](https://colab.research.google.com/), quindi scegli **File -> Carica notebook** e seleziona questo file `.ipynb`. Dopo la pubblicazione su GitHub, il link diretto avra forma `https://colab.research.google.com/github/OWNER/REPOSITORY/blob/main/edge_ai_mnist_tensorflow_teacher_solutions.ipynb`.

Per la latenza, seleziona **Runtime -> Change runtime type -> GPU**. MNIST funziona anche su CPU; la GPU rende solo l'addestramento e le misure piu rapidi.

Non applichiamo augmentation; convertiamo soltanto i pixel a `float32` nell'intervallo `[0, 1]`.

In [ ]:
import os
import time
import random
import tempfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BATCH_SIZE = 128
EPOCHS = 10
LEARNING_RATE = 1e-3
NUM_CLASSES = 10

## 1. Caricare MNIST

`tf.keras.datasets.mnist.load_data()` fornisce 60.000 immagini di training e 10.000 di test. I dati originali hanno forma `28×28`; aggiungiamo esplicitamente il canale finale per ottenere `28×28×1`, il formato atteso da `Conv2D`. Riserviamo 5.000 esempi alla validazione.

In [ ]:
# MNIST restituisce immagini 28 x 28 e etichette intere da 0 a 9.
(x_train_full, y_train_full), (x_test, y_test) = tf.keras.datasets.mnist.load_data()

# Scala i pixel e aggiunge il canale grayscale richiesto da Conv2D: (N, 28, 28, 1).
x_train_full = x_train_full.astype("float32")[..., np.newaxis] / 255.0
x_test = x_test.astype("float32")[..., np.newaxis] / 255.0
y_train_full = y_train_full.astype("int64")
y_test = y_test.astype("int64")

# Split deterministico: 5,000 esempi per validazione, 55,000 per training.
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(x_train_full))
val_idx = indices[:5_000]
train_idx = indices[5_000:]
x_train, y_train = x_train_full[train_idx], y_train_full[train_idx]
x_val, y_val = x_train_full[val_idx], y_train_full[val_idx]

# Crea pipeline batched e prefetch per un input efficiente.
train_ds = (
    tf.data.Dataset.from_tensor_slices((x_train, y_train))
    .shuffle(len(x_train), seed=SEED, reshuffle_each_iteration=True)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)
val_ds = tf.data.Dataset.from_tensor_slices((x_val, y_val)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
test_ds = tf.data.Dataset.from_tensor_slices((x_test, y_test)).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

print("Train:", x_train.shape, y_train.shape)
print("Val:  ", x_val.shape, y_val.shape)
print("Test: ", x_test.shape, y_test.shape)

In [ ]:
class_names = [str(digit) for digit in range(NUM_CLASSES)]

fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, image, label in zip(axes.ravel(), x_train[:10], y_train[:10]):
    # Rimuove il canale finale per visualizzare una sola mappa grayscale.
    ax.imshow(image.squeeze(-1), cmap="gray", vmin=0, vmax=1)
    ax.set_title(class_names[int(label)])
    ax.axis("off")
plt.tight_layout()
plt.show()

## 2. Esercizio - implementare il MLP

Implementa `Input(28, 28, 1) -> Flatten -> Dense(128, ReLU) -> Dense(64, ReLU) -> Dense(10 logits)`. L'output non ha attivazione perche la loss applica il softmax ai logits.

In [ ]:
def build_mlp():
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(28, 28, 1)),  # one grayscale channel
            tf.keras.layers.Flatten(),                 # 28 x 28 x 1 -> 784
            tf.keras.layers.Dense(128, activation="relu"),
            tf.keras.layers.Dense(64, activation="relu"),
            tf.keras.layers.Dense(NUM_CLASSES),         # raw logits for the loss
        ],
        name="dense_mlp",
    )

mlp = build_mlp()
mlp.summary()

### Self-check for the MLP

Run this cell after implementing the model. It checks the input/output shape and the expected parameter count.

In [ ]:
def check_mlp(model):
    # These checks catch a wrong input channel, output shape, or layer width.
    assert model.input_shape == (None, 28, 28, 1), model.input_shape
    assert model.output_shape == (None, NUM_CLASSES), model.output_shape
    assert model.count_params() == 109_386, (
        f"Unexpected parameter count: {model.count_params():,}"
    )
    print("MLP architecture looks correct.")
    print(f"Parameters: {model.count_params():,}")

check_mlp(mlp)

## 3. Esercizio - implementare la small CNN

Implementa `Input(28, 28, 1) -> Conv3×3(8, same, ReLU) -> MaxPool2×2 -> Conv3×3(16, same, ReLU) -> MaxPool2×2 -> Flatten -> Dense(32, ReLU) -> Dense(10 logits)`. Dopo i pool la mappa e `7×7×16`, quindi `Flatten` restituisce 784 feature.

In [ ]:
def build_cnn():
    return tf.keras.Sequential(
        [
            tf.keras.layers.Input(shape=(28, 28, 1)),  # MNIST grayscale input
            tf.keras.layers.Conv2D(8, 3, strides=1, padding="same", activation="relu", use_bias=True),
            tf.keras.layers.MaxPool2D(pool_size=2, strides=2),  # 28 x 28 -> 14 x 14
            tf.keras.layers.Conv2D(16, 3, strides=1, padding="same", activation="relu", use_bias=True),
            tf.keras.layers.MaxPool2D(pool_size=2, strides=2),  # 14 x 14 -> 7 x 7
            tf.keras.layers.Flatten(),                           # 7 x 7 x 16 = 784
            tf.keras.layers.Dense(32, activation="relu"),
            tf.keras.layers.Dense(NUM_CLASSES),                  # raw logits
        ],
        name="small_cnn",
    )

cnn = build_cnn()
cnn.summary()

### Self-check for the CNN

In [ ]:
def check_cnn(model):
    assert model.input_shape == (None, 28, 28, 1), model.input_shape
    assert model.output_shape == (None, NUM_CLASSES), model.output_shape
    assert model.count_params() == 26_698, (
        f"Unexpected parameter count: {model.count_params():,}"
    )

    # Probe the Flatten layer: two pools must yield 7 x 7 x 16 = 784.
    flatten_layers = [layer for layer in model.layers if isinstance(layer, tf.keras.layers.Flatten)]
    assert len(flatten_layers) == 1, "Expected exactly one Flatten layer."
    probe = tf.keras.Model(model.input, flatten_layers[0].output)
    features = probe(tf.zeros((1, 28, 28, 1)))
    assert features.shape[-1] == 784, f"Expected 784 features, got {features.shape[-1]}"

    print("CNN architecture looks correct.")
    print(f"Parameters: {model.count_params():,}")
    print("Flatten features:", int(features.shape[-1]))

check_cnn(cnn)

## 4. Compile and train both models

To keep the comparison controlled, use for both models:

- Adam, learning rate `1e-3`;
- sparse categorical cross-entropy **from logits**;
- the same batch size and number of epochs.

**Exercise:** instantiate both models, compile them, and train them.

In [ ]:
def compile_model(model):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=LEARNING_RATE),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
        metrics=["accuracy"],
    )
    return model

tf.keras.backend.clear_session()
tf.random.set_seed(SEED)

mlp = compile_model(build_mlp())
cnn = compile_model(build_cnn())

check_mlp(mlp)
check_cnn(cnn)

print("\nTraining MLP")
history_mlp = mlp.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=2,
)

print("\nTraining CNN")
history_cnn = cnn.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    verbose=2,
)

## 5. Compare training curves

Plot training and validation loss/accuracy for the two models. Look for:

- convergence speed;
- final validation accuracy;
- train/validation gap;
- signs of underfitting or overfitting.

In [ ]:
plot_histories(history_mlp, history_cnn)

## 6. Test accuracy

In [ ]:
mlp_test_loss, mlp_test_acc = test_accuracy(mlp)
cnn_test_loss, cnn_test_acc = test_accuracy(cnn)

print(f"MLP test loss:     {mlp_test_loss:.4f}")
print(f"MLP test accuracy: {mlp_test_acc:.4f}")
print(f"CNN test loss:     {cnn_test_loss:.4f}")
print(f"CNN test accuracy: {cnn_test_acc:.4f}")

## 7. Model size in memory and on disk

We report two related but different quantities:

1. **Parameter footprint** = number of stored parameter bytes. With float32, this is approximately `4 × #parameters`.
2. **Serialized weight-file size** = actual `.weights.h5` file size, which also includes file-format metadata.

This is **not** the full runtime memory footprint: activations, framework buffers, allocator overhead, and temporary workspaces are not included.

In [ ]:
def parameter_memory_bytes(model):
    total = 0
    for v in model.weights:
        total += int(np.prod(v.shape)) * int(v.dtype.size)
    return total

def saved_weights_size_bytes(model, filename):
    path = Path(tempfile.gettempdir()) / filename
    model.save_weights(path)
    return path.stat().st_size

mlp_param_bytes = parameter_memory_bytes(mlp)
cnn_param_bytes = parameter_memory_bytes(cnn)
mlp_file_bytes = saved_weights_size_bytes(mlp, "mlp.weights.h5")
cnn_file_bytes = saved_weights_size_bytes(cnn, "cnn.weights.h5")

print(f"MLP parameter footprint: {mlp_param_bytes / 2**20:.3f} MiB")
print(f"CNN parameter footprint: {cnn_param_bytes / 2**20:.3f} MiB")
print(f"MLP serialized weights:  {mlp_file_bytes / 2**20:.3f} MiB")
print(f"CNN serialized weights:  {cnn_file_bytes / 2**20:.3f} MiB")

## 8. Count MACs per image

We count MACs for **Conv2D** and **Dense** layers only.

- Conv2D: `Hout × Wout × Cout × Kh × Kw × Cin`
- Dense: `Nin × Nout`

Bias additions, ReLU, pooling, and data movement are excluded.

This matches the convention used in the lesson.

In [ ]:
def count_macs_keras(model, input_shape=(1, 28, 28, 1)):
    shape = tuple(input_shape)
    total_macs = 0
    for layer in model.layers:
        if isinstance(layer, tf.keras.layers.InputLayer):
            continue
        out_shape = tuple(layer.compute_output_shape(shape))
        if isinstance(layer, tf.keras.layers.Conv2D):
            _, h_out, w_out, c_out = out_shape
            k_h, k_w = layer.kernel_size
            total_macs += int(h_out * w_out * c_out * k_h * k_w * shape[-1])
        elif isinstance(layer, tf.keras.layers.Dense):
            total_macs += int(shape[-1] * layer.units)
        shape = out_shape
    return total_macs

mlp_macs = count_macs_keras(mlp)
cnn_macs = count_macs_keras(cnn)

print(f"MLP MACs/image: {mlp_macs:,}")
print(f"CNN MACs/image: {cnn_macs:,}")

assert mlp_macs == 109_184
assert cnn_macs == 307_648

### Controllo manuale dei MAC

Calcola i MAC prima di eseguire `count_macs_keras()`. Un MAC equivale a una moltiplicazione piu accumulo; bias, ReLU e pooling non sono inclusi.

**CNN**

- Conv1: `28 x 28 x 8 x 3 x 3 x 1 = 56,448`
- Conv2: `14 x 14 x 16 x 3 x 3 x 8 = 225,792`
- Dense32: `784 x 32 = 25,088`
- Dense10: `32 x 10 = 320`
- **Totale CNN = 307,648 MAC/image**

**MLP**

- Dense128: `784 x 128 = 100,352`
- Dense64: `128 x 64 = 8,192`
- Dense10: `64 x 10 = 640`
- **Totale MLP = 109,184 MAC/image**

## 9. Batch-1 inference latency

We benchmark a single image, after warm-up.

**Important:** latency depends on hardware, TensorFlow version, kernel selection, graph compilation, and synchronization. Use it as an empirical measurement, not as a hardware-independent property of the network.

Compare the two TensorFlow models on the **same Colab runtime**. Do not interpret a TensorFlow-vs-PyTorch latency difference as a pure architecture difference.

In [ ]:
x_one = x_test[:1]

mlp_latency = benchmark_latency_ms(mlp, x_one)
cnn_latency = benchmark_latency_ms(cnn, x_one)

print("MLP:", mlp_latency)
print("CNN:", cnn_latency)

## 10. Final comparison table

Run this cell after all previous measurements are available.

In [ ]:
def mib(n_bytes):
    return n_bytes / (2**20)

results = pd.DataFrame([
    {
        "model": "MLP",
        "test_accuracy": mlp_test_acc,
        "parameters": mlp.count_params(),
        "parameter_MiB": mib(mlp_param_bytes),
        "weight_file_MiB": mib(mlp_file_bytes),
        "MACs_per_image": mlp_macs,
        "latency_median_ms": mlp_latency["median_ms"],
        "latency_p95_ms": mlp_latency["p95_ms"],
    },
    {
        "model": "CNN",
        "test_accuracy": cnn_test_acc,
        "parameters": cnn.count_params(),
        "parameter_MiB": mib(cnn_param_bytes),
        "weight_file_MiB": mib(cnn_file_bytes),
        "MACs_per_image": cnn_macs,
        "latency_median_ms": cnn_latency["median_ms"],
        "latency_p95_ms": cnn_latency["p95_ms"],
    },
]).set_index("model")

display(results.style.format({
    "test_accuracy": "{:.4f}",
    "parameter_MiB": "{:.3f}",
    "weight_file_MiB": "{:.3f}",
    "MACs_per_image": "{:,.0f}",
    "latency_median_ms": "{:.3f}",
    "latency_p95_ms": "{:.3f}",
}))

## 11. Discussion questions

Write short answers based on **your measurements**, not on intuition alone.

1. The MLP and CNN have a similar MAC count. Why can their test accuracy still differ substantially?
2. Which model has more parameters? Where are most of those parameters located?
3. Does lower parameter count automatically imply lower latency? Explain using your measurement.
4. Why can two networks with similar MAC counts have different latency on the same GPU?
5. Why is the serialized file size not exactly equal to `4 × #parameters`?
6. If all weights were quantized from FP32 to INT8, what would you expect to happen to the **weight memory footprint**?
7. Which metrics here are properties of the **model**, and which depend strongly on the **runtime/hardware**?

## 12. Optional extension

Repeat the latency benchmark with batch sizes `1`, `8`, `32`, and `128`.

Plot:

- latency per batch;
- latency per image;
- throughput in images/s.

This separates **interactive latency** from **throughput**, an important distinction for edge deployment.

In [ ]:
def benchmark_batches(model, x_source, batch_sizes=(1, 8, 32, 128), warmup=20, runs=100):
    rows = []
    for bs in batch_sizes:
        x = tf.convert_to_tensor(x_source[:bs], dtype=tf.float32)
        stats = benchmark_latency_ms(model, x, warmup=warmup, runs=runs)
        batch_ms = stats["median_ms"]
        rows.append({
            "batch_size": bs,
            "batch_latency_ms": batch_ms,
            "ms_per_image": batch_ms / bs,
            "images_per_second": 1000.0 * bs / batch_ms,
        })
    return pd.DataFrame(rows)

print("MLP")
display(benchmark_batches(mlp, x_test))
print("CNN")
display(benchmark_batches(cnn, x_test))

---
### Controlli e riferimenti

Questi valori non dipendono dal training:

- MLP: **109,386 parametri**, **109,184 MAC/image**;
- CNN: **26,698 parametri**, **307,648 MAC/image**.

Accuratezza e latenza sono empiriche: riporta quelle ottenute dalla tua esecuzione.

### Manuali ufficiali

- [Keras: classificazione con MNIST](https://keras.io/examples/vision/mnist_convnet/)
- [Keras Sequential](https://keras.io/api/models/sequential/)
- [Keras Conv2D](https://keras.io/api/layers/convolution_layers/convolution2d/)
- [Guida Google Colab](https://colab.research.google.com/notebooks/intro.ipynb)

## 13. Task opzionale - estendere a CIFAR-10

Quando hai completato MNIST, prova CIFAR-10. Il dataset usa immagini RGB `32×32×3`: aggiorna input, visualizzazione e le forme dei layer. Mantieni loss, metriche e numero di epoche per confrontare il costo dell'architettura, non un setup diverso. La cella seguente e disattivata per mantenere MNIST come percorso principale.

In [ ]:
# OPTIONAL: set to True only after completing the MNIST experiment.
RUN_CIFAR10_EXTENSION = False

if RUN_CIFAR10_EXTENSION:
    # CIFAR-10 is RGB, hence the input shape is 32 x 32 x 3.
    (cifar_x_train, cifar_y_train), (cifar_x_test, cifar_y_test) = tf.keras.datasets.cifar10.load_data()
    cifar_x_train = cifar_x_train.astype("float32") / 255.0
    cifar_x_test = cifar_x_test.astype("float32") / 255.0
    cifar_y_train = cifar_y_train.squeeze().astype("int64")
    cifar_y_test = cifar_y_test.squeeze().astype("int64")

    def build_cifar_cnn():
        return tf.keras.Sequential(
            [
                tf.keras.layers.Input(shape=(32, 32, 3)),
                tf.keras.layers.Conv2D(8, 3, padding="same", activation="relu"),
                tf.keras.layers.MaxPool2D(2),  # 32 x 32 -> 16 x 16
                tf.keras.layers.Conv2D(16, 3, padding="same", activation="relu"),
                tf.keras.layers.MaxPool2D(2),  # 16 x 16 -> 8 x 8
                tf.keras.layers.Flatten(),     # 16 x 8 x 8 features
                tf.keras.layers.Dense(32, activation="relu"),
                tf.keras.layers.Dense(NUM_CLASSES),  # logits
            ],
            name="cifar10_cnn",
        )

    cifar_cnn = compile_model(build_cifar_cnn())
    cifar_history = cifar_cnn.fit(
        cifar_x_train, cifar_y_train, validation_split=0.1,
        batch_size=BATCH_SIZE, epochs=EPOCHS, verbose=2,
    )
    cifar_macs = count_macs_keras(cifar_cnn, input_shape=(1, 32, 32, 3))
    print("CIFAR-10 CNN MACs/image:", cifar_macs)